# AIC HCMC 2025 — Qwen3-ASR-1.7B transcript

**Before running: `Kernel → Restart & Clear Outputs`**

Vietnamese transcript using [Qwen3-ASR-1.7B](https://huggingface.co/Qwen/Qwen3-ASR-1.7B-hf)
(2B params, SOTA among open-source ASR, supports 30 languages incl. Vietnamese).

- Video: `L27_V010.mp4` (Kaggle dataset `tommr211/aic-batch1-videos`)
- Output: `/kaggle/working/L27_V010_transcript.txt`

**Internet must be enabled** (model download). GPU recommended.
Note: requires `transformers>=5.13.0` (upgraded in the setup cell).

## 1. Setup (upgrade transformers to 5.x)

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# Qwen3-ASR needs transformers >= 5.13.0 (Kaggle ships 4.x)
pip("transformers>=5.13.0", "accelerate")

import torch
import transformers
print("torch:", torch.__version__, "| cuda:", torch.version.cuda, "| gpu:", torch.cuda.is_available())
print("transformers:", transformers.__version__)

## 2. Extract 16 kHz mono audio

In [ ]:
VIDEO = "/kaggle/input/datasets/tommr211/aic-batch1-videos/Videos_L27_a/video/L27_V010.mp4"
WAV = "/kaggle/working/L27_V010.wav"

subprocess.run(
    ["ffmpeg", "-y", "-i", VIDEO, "-vn", "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", WAV],
    check=True, capture_output=True,
)
print("wav written:", WAV)

## 3. Transcribe with Qwen3-ASR-1.7B

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

model_id = "Qwen/Qwen3-ASR-1.7B-hf"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForMultimodalLM.from_pretrained(model_id, device_map="auto")
print("model on", model.device, "dtype", model.dtype)

inputs = processor.apply_transcription_request(
    audio=WAV,
    language="Vietnamese",
).to(model.device, model.dtype)

with torch.inference_mode():
    output_ids = model.generate(**inputs, max_new_tokens=2048)

generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]
transcript = processor.decode(generated_ids, return_format="transcription_only")[0].strip()
print("TRANSCRIPT:\n" + transcript)

## 4. Save transcript

In [ ]:
out_txt = "/kaggle/working/L27_V010_transcript.txt"
with open(out_txt, "w") as f:
    f.write(transcript + "\n")
print("saved:", out_txt)